<a href="https://colab.research.google.com/github/mysciz/deep-rl-class/blob/main/notebooks/unit3/DQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q learning
## Enviroment

In [2]:
!pip install gymnasium[atari]
!pip install gymnasium[accept-rom-license]

In [3]:
%%capture
!apt install python-opengl
!apt install xvfb
!pip3 install pyvirtualdisplay

## Display

In [4]:
import gymnasium as gym
from pyvirtualdisplay import Display
virtual_display = Display(visible=0, size=(1400, 900))
virtual_display.start()

In [5]:
## Data process

In [6]:
class PreprocessFrame(gym.ObservationWrapper):
    def __init__(self, env):
        super(PreprocessFrame, self).__init__(env)
        # 最终输出 84x84
        self.observation_space = gym.spaces.Box(low=0, high=255, shape=(84, 84), dtype=np.uint8)

    def observation(self, obs):
        # 裁剪掉上下得分区域
        img = obs[34:194, :, :]
        # 转灰度
        img = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
        # 缩放
        img = cv2.resize(img, (84, 84), interpolation=cv2.INTER_AREA)
        return img

def make_atari_env(env_id):
    env = gym.make(env_id, render_mode="rgb_array")
    env = PreprocessFrame(env)
    # 核心：将连续 4 帧堆叠在一起
    env = gym.wrappers.FrameStackObservation(env, 4)
    return env

## Buffer Replay




In [9]:
import collections
Experience = collections.namedtuple('Experience', ['state', 'action', 'reward', 'done', 'new_state'])

class ExperienceBuffer:
    def __init__(self, capacity):
        self.buffer = collections.deque(maxlen=capacity)

    def __len__(self):
        return len(self.buffer)

    def append(self, exp):
        self.buffer.append(exp)

    def sample(self, batch_size):
        indices = np.random.choice(len(self.buffer), batch_size, replace=False)
        states, actions, rewards, dones, next_states = zip(*[self.buffer[idx] for idx in indices])
        return np.array(states), np.array(actions), np.array(rewards, dtype=np.float32), \
               np.array(dones, dtype=bool), np.array(next_states)

## Deep Q network

In [13]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import random
import cv2
class DQNetwork(nn.Module):
    def __init__(self, n_actions):
        super(DQNetwork, self).__init__()
        # 输入维度 (4, 84, 84)
        self.conv = nn.Sequential(
            nn.Conv2d(4, 32, kernel_size=8, stride=4), nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, stride=1), nn.ReLU()
        )
        self.fc = nn.Sequential(
            nn.Linear(64 * 7 * 7, 512), nn.ReLU(),
            nn.Linear(512, n_actions)
        )

    def forward(self, x):
        # 归一化：将像素 0-255 转为 0-1
        x = x.float() / 255.0
        conv_out = self.conv(x).view(x.size()[0], -1)
        return self.fc(conv_out)

## Train

In [14]:
import ale_py
# 核心参数
ENV_NAME = "ALE/Breakout-v5" # 这是 Gymnasium 最新版推荐的 ID
EPSILON_START = 1.0
EPSILON_FINAL = 0.02
EPS_DECAY = 100000
TARGET_UPDATE_FREQ = 1000
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前使用的设备: {device}")
# 初始化
env = make_atari_env(ENV_NAME)
net = DQNetwork(env.action_space.n).to(device)
tgt_net = DQNetwork(env.action_space.n).to(device)
tgt_net.load_state_dict(net.state_dict())
buffer = ExperienceBuffer(20000)
optimizer = optim.Adam(net.parameters(), lr=1e-4)

state, _ = env.reset()
total_rewards = []
cur_reward = 0
frame_idx = 0

print(">>> 开始收集经验并训练...")

while frame_idx < 500000:
    frame_idx += 1
    epsilon = max(EPSILON_FINAL, EPSILON_START - frame_idx / EPS_DECAY)

    # 动作选择
    if random.random() < epsilon:
        action = env.action_space.sample()
    else:
        state_v = torch.from_numpy(np.array([state])).to(device)
        action = net(state_v).argmax().item()

    next_state, reward, term, trunc, _ = env.step(action)
    done = term or trunc
    buffer.append(Experience(state, action, reward, done, next_state))
    state = next_state
    cur_reward += reward

    if done:
        total_rewards.append(cur_reward)
        state, _ = env.reset()
        if len(total_rewards) % 10 == 0:
            print(f"帧: {frame_idx} | 10局均分: {np.mean(total_rewards[-10:]):.2f}")
        cur_reward = 0

    # 经验回放训练
    if len(buffer) > 10000 and frame_idx % 4 == 0:
        s, a, r, d, ns = buffer.sample(32)
        s_v = torch.tensor(s).to(device)
        a_v = torch.tensor(a).to(device).long()
        r_v = torch.tensor(r).to(device)
        d_v = torch.tensor(d).to(device)
        ns_v = torch.tensor(ns).to(device)

        q = net(s_v).gather(1, a_v.unsqueeze(-1)).squeeze(-1)
        with torch.no_grad():
            max_ns_q = tgt_net(ns_v).max(1)[0]
            max_ns_q[d_v] = 0.0
            t_q = r_v + 0.99 * max_ns_q

        loss = nn.MSELoss()(q, t_q)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    if frame_idx % TARGET_UPDATE_FREQ == 0:
        tgt_net.load_state_dict(net.state_dict())

当前使用的设备: cuda
>>> 开始收集经验并训练...
帧: 1923 | 10局均分: 1.20
帧: 3667 | 10局均分: 1.10
帧: 5457 | 10局均分: 1.10
帧: 7130 | 10局均分: 0.80
帧: 8936 | 10局均分: 1.10
帧: 10731 | 10局均分: 0.90
帧: 12638 | 10局均分: 1.50
帧: 14479 | 10局均分: 1.30
帧: 16402 | 10局均分: 1.20
帧: 18313 | 10局均分: 1.30
帧: 20606 | 10局均分: 2.50
帧: 22578 | 10局均分: 1.60
帧: 24654 | 10局均分: 2.00
帧: 26593 | 10局均分: 1.30
帧: 28392 | 10局均分: 1.30
帧: 30278 | 10局均分: 1.30
帧: 32281 | 10局均分: 1.50
帧: 34221 | 10局均分: 1.70
帧: 36352 | 10局均分: 2.10
帧: 38460 | 10局均分: 2.20
帧: 40965 | 10局均分: 3.40
帧: 43158 | 10局均分: 2.20
帧: 45352 | 10局均分: 2.30
帧: 47544 | 10局均分: 2.30
帧: 49803 | 10局均分: 2.20
帧: 52314 | 10局均分: 3.60
帧: 54882 | 10局均分: 4.00
帧: 57315 | 10局均分: 3.10
帧: 59619 | 10局均分: 3.50
帧: 62344 | 10局均分: 4.70
帧: 64498 | 10局均分: 2.90
帧: 67019 | 10局均分: 3.60
帧: 69749 | 10局均分: 4.50
帧: 72465 | 10局均分: 4.50
帧: 75669 | 10局均分: 5.90
帧: 78449 | 10局均分: 4.70
帧: 81397 | 10局均分: 4.40
帧: 84129 | 10局均分: 4.60
帧: 87143 | 10局均分: 4.70
帧: 89609 | 10局均分: 4.20
帧: 92224 | 10局均分: 4.60
帧: 95649 | 10局均分: 5.80
帧: 98223

In [19]:
from gymnasium.utils.save_video import save_video
import os
from gymnasium import wrappers
def record_evaluation_video(net, env_id, file_name="ai_demo"):
    # 创建专门用于渲染的评估环境
    eval_env = gym.make(env_id, render_mode="rgb_array_list")
    eval_env = PreprocessFrame(eval_env)
    eval_env = wrappers.FrameStackObservation(eval_env, stack_size=4)

    state, _ = eval_env.reset()
    done = False
    total_reward = 0

    net.eval() # 切换到评估模式（关闭 Dropout 等）

    while not done:
        # AI 决策 (不使用 epsilon 探索，直接选最优)
        state_v = torch.from_numpy(np.array([state])).to(device)
        with torch.no_grad():
            action = net(state_v).argmax().item()

        state, reward, term, trunc, _ = eval_env.step(action)
        total_reward += reward
        done = term or trunc

    # 保存视频到 'videos' 文件夹
    save_video(eval_env.render(), "videos", fps=30, name_prefix=file_name)
    eval_env.close()
    print(f">>> 演示完成！总得分: {total_reward}")
    print(f">>> 视频已保存在: videos/{file_name}-episode-0.mp4")

# 调用函数（确保 net 和 ENV_NAME 已定义）
record_evaluation_video(net, ENV_NAME)

>>> 演示完成！总得分: 8.0
>>> 视频已保存在: videos/ai_demo-episode-0.mp4


In [20]:
from IPython.display import HTML
from base64 import b64encode

# 读取生成的视频文件
video_path = 'videos/ai_demo-episode-0.mp4'
video_file = open(video_path, "r+b").read()

# 将视频转码为 Base64 格式
video_url = f"data:video/mp4;base64,{b64encode(video_file).decode()}"

# 在页面中嵌入 HTML5 播放器
HTML(f"""
<video width="600" controls>
      <source src="{video_url}" type="video/mp4">
</video>
""")